In [1]:
# Google Colab / local setup
import os, sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODE_ROOT = Path("/content/drive/MyDrive/msc project/code")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
        str(CODE_ROOT), 'catboost==1.2.10', 'rapidfuzz==3.14.3',
    ])
else:
    CODE_ROOT = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    )
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
CODE = CODE_ROOT

# 02d — All-model raw sentiment feature comparison

This notebook continues Notebook 02c and asks one isolated question: **does adding sentiment improve otherwise unchanged model outputs?** Notebook 02c established the sentiment provenance, scorer values, causal alignment and exact six-column schema. Here, nine fixed model-family baselines use the same raw 15-minute execution, so differences within each Model/DZ come only from the feature layer.

## Handoff from Notebooks 02b and 02c

Notebook 02b passes **Fixed baseline candidate 0**, DZ55, DZ65 and DZ75, the fixed **180-day** training history and the unchanged 24-feature schema. Notebook 02c passes the matched DeBERTa/LLM feature contract. The cross-model extension uses candidate 0 for Logistic Regression, Decision Tree, Random Forest, Linear SVM, XGBoost, CatBoost, MLP, LSTM and GRU: each is its declared project baseline, not a newly tuned model. No execution policy is transferred.

## Methodology

Four feature arms share the same 24 price/order-flow/positioning predictors; only the sentiment layer changes.

| Arm | Added columns | Definition |
|---|---|---|
| No sentiment | 0 | Market-only control |
| DeBERTa | 6 | Encoder sentiment, own weighting and shared news/event controls |
| LLM-matched | 6 | Unweighted LLM scalar with the same feature types |
| LLM-full | 9 | Own-weighted sentiment plus relevance, impact and asset channels |

LLM-full tests the complete feature layer, not a scalar-only substitution.

Classification uses five 2024 blocked folds with a label-tail trim; each July-2025 model uses the preceding 180 days and remains fixed through March 2026.
Raw directional predictions enter next-M15 open, hold one bar, prohibit overlap and deduct 5 bps per side; no confidence gate or TP/SL is applied.

### Evaluation periods

2024 supplies classification diagnostics, H1-2025 supplies the July fit history, and July-2025–March-2026 supplies descriptive forward economics; Q2 is excluded from this comparison.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

CODE_ROOT = Path.cwd().resolve()
if CODE_ROOT.name == "notebooks":
    CODE_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from experiments.all_model_sentiment_raw import POLICY_COLUMNS
from experiments.all_model_sentiment_scoreboard import ARM_LABELS, build_scoreboards
from experiments.notebook02b_handoff import HANDOFF_PATH, load_notebook02b_handoff
from experiments.raw_hold_control import MODEL_LABELS

handoff = load_notebook02b_handoff(HANDOFF_PATH)
tables = build_scoreboards()
classification = tables["classification"]
economics = tables["economics"]

assert handoff["candidate_id"] == 0
assert handoff["training_history_days"] == 180
assert len(classification) == len(economics) == 108
assert economics["candidate_id"].eq(0).all()
assert set(economics["width_bps"].astype(int)) == {55, 65, 75}
assert economics["lookback_days"].eq(180).all()
assert economics["hold_minutes"].eq(15).all()
assert not POLICY_COLUMNS.intersection(economics.columns)
assert economics["period_end"].max() <= pd.Timestamp("2026-04-01", tz="UTC")

## Results

Table 1 diagnoses classification on unchanged 2024 blocking folds and includes the matched raw-forward Sortino only to define a common row order. Table 2 reports the matched raw forward economics. Every row remains visible; all result tables are ordered by Sortino from highest to lowest.

### 1. Classification diagnostics

Overall macro-F1 summarises all three classes; Robust F1 is the weakest-regime fold score and penalises performance that depends on one market regime.

In [3]:
from IPython.display import Markdown
classification_table = (
    classification.merge(
        economics[["sentiment_arm", "model_name", "width_bps", "sortino"]],
        on=["sentiment_arm", "model_name", "width_bps"],
        validate="one_to_one",
    )[["model_name", "Arm", "width_bps", "overall_f1", "robust_f1", "sortino"]]
    .assign(model_name=lambda frame: frame["model_name"].map(MODEL_LABELS))
    .rename(columns={
        "model_name": "Model",
        "width_bps": "DZ",
        "overall_f1": "Overall macro-F1",
        "robust_f1": "Robust F1",
        "sortino": "Sortino",
    })
    .sort_values(["Sortino", "Robust F1"], ascending=False, ignore_index=True)
)
display(Markdown('Method: Nine families and three widths use unchanged 2024 blocked tests for each sentiment arm.'))
display(classification_table.round({"Overall macro-F1": 3, "Robust F1": 3, "Sortino": 3}))
display(Markdown('Predictive gains vary by model and do not establish a universal sentiment advantage.'))

Method: Nine families and three widths use unchanged 2024 blocked tests for each sentiment arm.

,Model,Arm,DZ,Overall macro-F1,Robust F1,Sortino
0,Linear SVM,No sentiment,75,0.345,0.326,1.838
1,Linear SVM,LLM-full,75,0.332,0.326,1.210
2,Linear SVM,LLM-full,65,0.330,0.324,-0.091
3,Linear SVM,No sentiment,65,0.344,0.324,-0.162
4,Linear SVM,LLM-full,55,0.332,0.320,-0.529
5,Linear SVM,LLM-matched,75,0.334,0.326,-1.249
6,Linear SVM,No sentiment,55,0.346,0.320,-1.326
7,Linear SVM,LLM-matched,65,0.326,0.320,-1.682
8,Linear SVM,LLM-matched,55,0.326,0.320,-1.967
9,XGBoost,DeBERTa,75,0.369,0.323,-5.964


Predictive gains vary by model and do not establish a universal sentiment advantage.

### 2. Raw frozen-forward economics

Every row uses direct class predictions and the same 15-minute hold. Differences within a matched Model/DZ therefore come only from the added sentiment features and their effect on directional predictions.

In [4]:
from IPython.display import Markdown
economic_table = (
    economics[[
        "model_name", "Arm", "width_bps", "trades", "gross_return",
        "net_return", "sortino", "sharpe", "positive_months",
    ]]
    .assign(model_name=lambda frame: frame["model_name"].map(MODEL_LABELS))
    .rename(columns={
        "model_name": "Model",
        "width_bps": "DZ",
        "trades": "Trades",
        "gross_return": "Gross return",
        "net_return": "Net return",
        "sortino": "Sortino",
        "sharpe": "Sharpe",
        "positive_months": "Positive months",
    })
    .sort_values(["Sortino", "Sharpe", "Net return"], ascending=False, ignore_index=True)
)
shown = economic_table.copy()
shown[["Sortino", "Sharpe"]] = shown[["Sortino", "Sharpe"]].round(3)
shown["Gross return"] = shown["Gross return"].map(lambda value: f"{100 * value:.2f}%")
shown["Net return"] = shown["Net return"].map(lambda value: f"{100 * value:.2f}%")
display(Markdown('Method: Frozen predictions use a raw 15-minute hold and identical costs within every Model/DZ pair.'))
display(shown)
display(Markdown('The strongest raw row is the no-sentiment SVM control.'))

leader = economics.sort_values(["sortino", "sharpe", "net_return"], ascending=False).iloc[0]

paired = economics.pivot(
    index=["model_name", "width_bps"], columns="sentiment_arm", values="net_return"
)
cells = len(paired)
for arm, label in (("classic", "DeBERTa"), ("llm", "LLM-matched"), ("llm_full", "LLM-full")):
    delta = paired[arm] - paired["none"]

for a, b, label in (("llm_full", "classic", "LLM-full vs DeBERTa (layer vs layer)"),
                    ("llm_full", "llm", "LLM-full vs LLM-matched (structure vs scalar)")):
    delta = paired[a] - paired[b]

per_trade = economics.assign(
    gross_bps=economics["gross_return"] / economics["trades"].clip(lower=1) * 1e4)
display(Markdown('Method: Per-trade gross return and total activity are averaged within each sentiment arm.'))
display(per_trade.groupby("sentiment_arm")[["gross_bps", "net_return", "trades"]]
        .median().round(3))
display(Markdown('Changes in activity must be distinguished from changes in per-trade quality.'))

best_per_arm = (per_trade.sort_values("sortino", ascending=False)
                .groupby("sentiment_arm").head(1)
                .sort_values("sortino", ascending=False))
shown = best_per_arm[["sentiment_arm", "model_name", "width_bps", "trades",
                      "gross_return", "net_return", "sortino", "sharpe",
                      "positive_months", "gross_bps"]].copy()
shown["sentiment_arm"] = shown["sentiment_arm"].map(ARM_LABELS)
shown["model_name"] = shown["model_name"].map(MODEL_LABELS)
for column in ("gross_return", "net_return"):
    shown[column] = shown[column].map(lambda value: f"{100 * value:.2f}%")
display(Markdown('Method: The best descriptive raw-forward row is retained separately for each feature arm.'))
display(shown.rename(columns={
    "sentiment_arm": "Arm", "model_name": "Model", "width_bps": "DZ",
    "trades": "Trades", "gross_return": "Gross", "net_return": "Net",
    "sortino": "Sortino", "sharpe": "Sharpe", "positive_months": "Positive months",
    "gross_bps": "Gross bps/trade"}).round(3).reset_index(drop=True))
display(Markdown("An arm's strongest row is not evidence of an overall sentiment improvement."))

Method: Frozen predictions use a raw 15-minute hold and identical costs within every Model/DZ pair.

,Model,Arm,DZ,Trades,Gross return,Net return,Sortino,Sharpe,Positive months
0,Linear SVM,No sentiment,75,40,11.51%,7.51%,1.838,1.201,6
1,Linear SVM,LLM-full,75,26,7.02%,4.42%,1.210,0.792,3
2,Linear SVM,LLM-full,65,42,3.80%,-0.40%,-0.091,-0.066,2
3,Linear SVM,No sentiment,65,67,5.84%,-0.86%,-0.162,-0.121,5
4,Linear SVM,LLM-full,55,81,4.68%,-3.42%,-0.529,-0.412,5
5,Linear SVM,LLM-matched,75,57,-2.39%,-8.09%,-1.249,-1.018,3
6,Linear SVM,No sentiment,55,105,0.95%,-9.55%,-1.326,-1.047,3
7,Linear SVM,LLM-matched,65,99,-2.32%,-12.22%,-1.682,-1.379,4
8,Linear SVM,LLM-matched,55,135,-2.14%,-15.64%,-1.967,-1.604,3
9,XGBoost,DeBERTa,75,585,-3.51%,-62.01%,-5.964,-4.736,1


The strongest raw row is the no-sentiment SVM control.

Method: Per-trade gross return and total activity are averaged within each sentiment arm.

,gross_bps,net_return,trades
sentiment_arm,,,
classic,-0.572,-6.945,6684.0
llm,-0.720,-6.212,5831.0
llm_full,-0.306,-6.361,6071.0
none,-0.757,-4.897,4647.0


Changes in activity must be distinguished from changes in per-trade quality.

Method: The best descriptive raw-forward row is retained separately for each feature arm.

,Arm,Model,DZ,Trades,Gross,Net,Sortino,Sharpe,Positive months,Gross bps/trade
0,No sentiment,Linear SVM,75,40,11.51%,7.51%,1.838,1.201,6,28.778
1,LLM-full,Linear SVM,75,26,7.02%,4.42%,1.210,0.792,3,27.009
2,LLM-matched,Linear SVM,75,57,-2.39%,-8.09%,-1.249,-1.018,3,-4.184
3,DeBERTa,XGBoost,75,585,-3.51%,-62.01%,-5.964,-4.736,1,-0.600


An arm's strongest row is not evidence of an overall sentiment improvement.

### 3. Five strongest model families

Each model family first keeps its best Sentiment/DZ row. The five distinct models are then ranked by Sortino, with Sharpe and net return as tie-breakers; this prevents several configurations of one model from occupying the whole shortlist.

In [5]:
from IPython.display import Markdown
top_five = (
    economics
    .sort_values(["sortino", "sharpe", "net_return"], ascending=False)
    .drop_duplicates("model_name", keep="first")
    .head(5)
    [[
        "model_name", "Arm", "width_bps", "trades", "gross_return",
        "net_return", "sortino", "sharpe", "positive_months",
    ]]
    .assign(model_name=lambda frame: frame["model_name"].map(MODEL_LABELS))
    .rename(columns={
        "model_name": "Model",
        "width_bps": "DZ",
        "trades": "Trades",
        "gross_return": "Gross return",
        "net_return": "Net return",
        "sortino": "Sortino",
        "sharpe": "Sharpe",
        "positive_months": "Positive months",
    })
)
assert len(top_five) == 5 and top_five["Model"].nunique() == 5
shown_top = top_five.copy()
shown_top[["Sortino", "Sharpe"]] = shown_top[["Sortino", "Sharpe"]].round(3)
shown_top["Gross return"] = shown_top["Gross return"].map(lambda value: f"{100 * value:.2f}%")
shown_top["Net return"] = shown_top["Net return"].map(lambda value: f"{100 * value:.2f}%")
display(Markdown('Method: Each family keeps its strongest raw-forward Arm/DZ row, ranked by Sortino, Sharpe and Net.'))
display(shown_top)
display(Markdown('The leading raw families do not establish a broad profitable sentiment effect.'))

Method: Each family keeps its strongest raw-forward Arm/DZ row, ranked by Sortino, Sharpe and Net.

,Model,Arm,DZ,Trades,Gross return,Net return,Sortino,Sharpe,Positive months
103,Linear SVM,No sentiment,75,40,11.51%,7.51%,1.838,1.201,6
104,XGBoost,DeBERTa,75,585,-3.51%,-62.01%,-5.964,-4.736,1
72,CatBoost,DeBERTa,75,273,-20.60%,-47.90%,-6.091,-5.014,1
96,Random Forest,DeBERTa,75,2437,-28.67%,-272.37%,-16.599,-13.210,0
87,Logistic Regression,No sentiment,75,4479,0.55%,-447.35%,-22.733,-18.388,0


The leading raw families do not establish a broad profitable sentiment effect.

## Decision and handoff

Two rows in the whole table are positive after costs, and both are Linear SVM at DZ75: the
**no-sentiment control** (40 trades, +7.51% net, 1.838 Sortino) and **LLM-full** (26 trades,
+4.42% net, 1.210 Sortino). Both sit below the 50-event floor Notebook 02e applies, so
neither is promotable on its own.

The aggregate verdict is unchanged from the no-sentiment baseline: **no sentiment arm beats
the control**, each winning only 8–9 of 27 matched cells with a negative median delta.
Sentiment is not promoted.

What *is* new is the ordering **within** the sentiment layer, and it is the part the novelty
claim turns on. LLM-full carries the strongest gross edge per trade of any arm — the
quantity that decides everything against a 10 bps round trip — and beats DeBERTa in 16 of 27
cells. Reducing the same LLM to a single scalar (LLM-matched) gives that back. So the
structured output is doing real work; it is simply not enough work to clear costs.

Notebook 02d makes no policy choice. Notebook 02e calibrates execution policy on exactly
these predictions and keeps the no-sentiment control alongside every sentiment arm.
Q2-2026 is evaluated separately in Notebook 07.